# 2층 신경망: 순전파와 역전파 손계산 검증

[Step 5 이슈 #8](https://github.com/ntt65/A2_1/issues/8)의 고정 예제다. 입력 2개 → 은닉 2개 → 출력 1개이며 두 층의 활성함수는 sigmoid, 손실은 **단일 샘플** Binary Cross-Entropy(BCE)다.

먼저 연쇄 법칙과 독립 손계산 기준값을 정리하고 NumPy 결과를 비교한다. 마지막에는 가중치·편향 **9개 전체**에 중심차분을 적용한다. 자동 미분을 사용하지 않는다.

- 손계산 비교: 원소별 `atol=5e-5`, `rtol=0`, 소수점 네 자리 표시도 일치.
- 매개변수 gradient 교차 검증: `h=1e-5`, 최대 절대오차 `<=1e-6`.
- 새 커널에서 위에서 아래로 실행한다. 중간값은 반올림하지 않고 표시할 때만 네 자리로 반올림한다.

In [1]:
import json
import os
from pathlib import Path
import sys

import numpy as np

repo_root = Path.cwd().resolve()
if not (repo_root / "src" / "utils" / "math.py").is_file():
    candidate = repo_root.parent
    if repo_root.name == "notebooks" and (candidate / "src" / "utils" / "math.py").is_file():
        repo_root = candidate
    else:
        raise RuntimeError("Run this notebook from the A2_1 root or its notebooks directory")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.utils.math import sigmoid
from src.utils.random import set_seed
from src.calculus import numerical_gradient

set_seed(42)
output_dir = Path(os.environ.get("A2_1_OUTPUT_DIR", str(repo_root / "outputs"))).resolve()
output_dir.mkdir(parents=True, exist_ok=True)
print(f"NumPy {np.__version__}; float64; seed=42; fixed input (no random sampling)")

NumPy 2.2.6; float64; seed=42; fixed input (no random sampling)


## 1. 입력과 곱 방향

| 기호 | 값 | shape |
|---|---|---|
| $x$ | $[1,0]^T$ | `(2,)` |
| $W_1$ | $\begin{bmatrix}.1&.2\\.3&.4\end{bmatrix}$ | `(2,2)` |
| $b_1$ | $[0,0]^T$ | `(2,)` |
| $W_2$ | $[.5,.6]$ | `(2,)` |
| $b_2,y$ | $0,1$ | scalar `()` |

NumPy의 1차원 배열에는 행/열 축이 별도로 없지만 수식에서는 $x$를 열벡터로 본다. 첫 층은 반드시 **`W1 @ x + b1`**이다. 출력은 `W2 @ a1 + b2`이며 배치 평균 계수는 없다.

In [2]:
x = np.array([1.0, 0.0], dtype=np.float64)
W1 = np.array([[0.1, 0.2], [0.3, 0.4]], dtype=np.float64)
b1 = np.zeros(2, dtype=np.float64)
W2 = np.array([0.5, 0.6], dtype=np.float64)
b2 = np.float64(0.0)
y_true = np.float64(1.0)

assert x.shape == (2,) and W1.shape == (2, 2) and b1.shape == (2,)
assert W2.shape == (2,) and b2.shape == () and y_true.shape == ()

## 2. 순전파를 직접 계산하기

$$\sigma(z)=\frac{1}{1+e^{-z}},\qquad \sigma'(z)=\sigma(z)(1-\sigma(z)).$$

1. $z_1=W_1x+b_1=[.1,.3]^T$ — `(2,)`.
2. $a_1=\sigma(z_1)=[0.5249791874789399861,\;0.5744425168116589872]^T$ — `(2,)`.
3. $z_2=W_2a_1+b_2=.5a_{1,1}+.6a_{1,2}=0.60715510382646538534$ — `()`.
4. $\hat y=\sigma(z_2)=0.64729156999461775097$ — `()`.
5. $L=-[y\ln\hat y+(1-y)\ln(1-\hat y)]=-\ln\hat y=0.43495843685146062506$ — `()`.

로그는 자연로그다. 현재 표본은 $y=1$이고 $0<\hat y<1$이므로 위 확률 표현이 유효하다. 일반적으로 확률이 float64에서 0 또는 1로 포화되면 `log(0)` 문제가 생긴다. $\hat y=\sigma(z_2)$를 대입하여 정리한 동치식

$$L=\log(1+e^{z_2})-yz_2=\operatorname{logaddexp}(0,z_2)-yz_2$$

을 실제 손실 계산에 사용한다. 확률을 임의로 clipping한 다른 함수를 만들지 않는다.

아래 기준 상수는 위 스칼라 식을 독립적으로 고정밀 계산해 옮긴 값이다. NumPy 순전파 결과를 복사하여 기대값을 만드는 방식이 아니다.

In [3]:
hand_reference = {
    "z1": np.array([0.1, 0.3]),
    "a1": np.array([0.5249791874789399861, 0.5744425168116589872]),
    "z2": np.float64(0.60715510382646538534),
    "y_pred": np.float64(0.64729156999461775097),
    "L": np.float64(0.43495843685146062506),
    "dL_dy_pred": np.float64(-1.5448988467566710784),
    "dL_dz2": np.float64(-0.35270843000538224903),
    "dL_dW2": np.array([-0.18516458500119814929, -0.20261071823298063973]),
    "dL_da1": np.array([-0.17635421500269112452, -0.21162505800322934942]),
    "dL_dz1": np.array([-0.04397851580869701354, -0.05173350439092561391]),
    "dL_dW1": np.array([[-0.04397851580869701354, 0.0], [-0.05173350439092561391, 0.0]]),
    "dL_db1": np.array([-0.04397851580869701354, -0.05173350439092561391]),
    "dL_db2": np.float64(-0.35270843000538224903),
}

In [4]:
z1 = W1 @ x + b1
a1 = sigmoid(z1)
z2 = W2 @ a1 + b2
y_pred = sigmoid(z2)
loss = np.logaddexp(0.0, z2) - y_true * z2
probability_loss = -(y_true*np.log(y_pred) + (1.0-y_true)*np.log1p(-y_pred))
np.testing.assert_allclose(loss, probability_loss, atol=1e-15, rtol=0)

forward = {"z1": z1, "a1": a1, "z2": z2, "y_pred": y_pred, "L": loss}
for name, value in forward.items():
    print(f"{name:8} shape={str(np.shape(value)):6} value={value}")

z1       shape=(2,)   value=[0.1 0.3]
a1       shape=(2,)   value=[0.52497919 0.57444252]
z2       shape=()     value=0.6071551038264654
y_pred   shape=()     value=0.6472915699946178
L        shape=()     value=0.43495843685146063


## 3. 연쇄 법칙으로 역전파 유도

출력층에서 입력층 순서로 진행한다. $\odot$는 원소별 곱이다.

**① 예측값에 대한 손실 미분 — scalar `()`**

$$\frac{\partial L}{\partial\hat y}=-\frac{y}{\hat y}+\frac{1-y}{1-\hat y}=-\frac{1}{\hat y}=-1.5448988467566710784.$$

**② 출력 logit에 대한 미분 — scalar `()`**

$$\delta_2=\frac{\partial L}{\partial z_2}
=\frac{\partial L}{\partial\hat y}\hat y(1-\hat y)
=\hat y-y=-0.35270843000538224903.$$

**③ 출력 가중치에 대한 미분 — `(2,)`**

$$\frac{\partial L}{\partial W_2}=\delta_2 a_1
=[-0.18516458500119814929,-0.20261071823298063973].$$

**④ 은닉 활성값에 대한 미분 — `(2,)`**

$$\frac{\partial L}{\partial a_1}=\delta_2 W_2
=[-0.17635421500269112452,-0.21162505800322934942].$$

**⑤ 은닉 logit에 대한 미분 — `(2,)`**

$$\delta_1=\frac{\partial L}{\partial z_1}
=(\delta_2 W_2)\odot a_1\odot(1-a_1)
=[-0.04397851580869701354,-0.05173350439092561391].$$

**⑥ 첫 층 가중치에 대한 미분 — `(2,2)`**

$$\frac{\partial L}{\partial W_{1,ij}}=\delta_{1,i}x_j,\qquad
\frac{\partial L}{\partial W_1}=\delta_1x^T
=\begin{bmatrix}-0.04397851580869701354&0\\-0.05173350439092561391&0\end{bmatrix}.$$

따라서 NumPy에서는 **`np.outer(delta1, x)`**다. 이 고정 입력의 두 번째 성분이0이므로 두 번째 열은0이다.

편향은 덧셈의 미분이1이므로 $\partial L/\partial b_2=\delta_2$ — `()`, $\partial L/\partial b_1=\delta_1$ — `(2,)`다.

`z2`의 네 자리 표시는 **0.6072**, 두 번째 `delta1`은 **−0.0517**이다. 원문 참고 예시의 0.6071/−0.0513을 정답으로 복사하지 않는다([D12](../docs/DECISIONS.md)).

In [5]:
dL_dy_pred = -y_true/y_pred + (1.0-y_true)/(1.0-y_pred)
delta2_from_chain = dL_dy_pred * y_pred * (1.0-y_pred)
delta2 = y_pred-y_true
np.testing.assert_allclose(delta2_from_chain, delta2, atol=1e-15, rtol=0)
dW2 = delta2*a1
da1 = delta2*W2
delta1 = da1*a1*(1.0-a1)
dW1 = np.outer(delta1, x)
db1 = delta1.copy()
db2 = np.asarray(delta2).copy()

backward = {
    "dL_dy_pred": dL_dy_pred, "dL_dz2": delta2, "dL_dW2": dW2,
    "dL_da1": da1, "dL_dz1": delta1, "dL_dW1": dW1,
    "dL_db1": db1, "dL_db2": db2,
}
assert dW2.shape == (2,) and da1.shape == (2,) and delta1.shape == (2,)
assert dW1.shape == (2, 2) and db1.shape == (2,) and db2.shape == ()
np.testing.assert_array_equal(dW1[:, 1], np.zeros(2))

## 4. 손계산과 NumPy 비교

13개 체크포인트의 shape·유한성·원소별 오차를 검사한다. 숫자를 네 자리로 잘라 다음 계산에 재사용하지 않는다.
원래 배열의 `-0.0`과 `0.0`은 같은 수치다. **표시에만** 반올림된0의 부호를 통일하여 `-0.0000`이라는 문자열 차이가 잘못된 불일치를 만들지 않게 한다. 실제 gradient 배열은 변경하지 않는다.

In [6]:
def four_decimal_strings(value):
    """Return shape-preserving strings rounded to four decimals; display signed zero as 0.0000.

    Input is finite numeric data of any shape; this display-only function does
    not modify it. The output string ndarray has the same shape, including ().
    """
    array = np.asarray(value, dtype=np.float64)
    if not np.all(np.isfinite(array)):
        raise ValueError("display values must be finite")
    texts = [f"{0.0 if round(float(item), 4) == 0 else float(item):.4f}" for item in array.flat]
    return np.array(texts).reshape(array.shape)


computed = {**forward, **backward}
hand_errors = {}
assert set(computed) == set(hand_reference)
for name, expected in hand_reference.items():
    actual = np.asarray(computed[name])
    assert actual.shape == np.shape(expected), name
    assert np.all(np.isfinite(actual)), name
    np.testing.assert_allclose(actual, expected, atol=5e-5, rtol=0, err_msg=name)
    actual_text = four_decimal_strings(actual)
    expected_text = four_decimal_strings(expected)
    np.testing.assert_array_equal(actual_text, expected_text, err_msg=name)
    hand_errors[name] = float(np.max(np.abs(actual-expected)))
    print(f"{name:12} shape={str(actual.shape):6} hand={expected_text} NumPy={actual_text}")

max_hand_absolute_error = max(hand_errors.values())
four_decimal_match = True
print(f"All 13 hand-reference checkpoints agree; maximum absolute error={max_hand_absolute_error:.3e}")

z1           shape=(2,)   hand=['0.1000' '0.3000'] NumPy=['0.1000' '0.3000']
a1           shape=(2,)   hand=['0.5250' '0.5744'] NumPy=['0.5250' '0.5744']
z2           shape=()     hand=0.6072 NumPy=0.6072
y_pred       shape=()     hand=0.6473 NumPy=0.6473
L            shape=()     hand=0.4350 NumPy=0.4350
dL_dy_pred   shape=()     hand=-1.5449 NumPy=-1.5449
dL_dz2       shape=()     hand=-0.3527 NumPy=-0.3527
dL_dW2       shape=(2,)   hand=['-0.1852' '-0.2026'] NumPy=['-0.1852' '-0.2026']
dL_da1       shape=(2,)   hand=['-0.1764' '-0.2116'] NumPy=['-0.1764' '-0.2116']
dL_dz1       shape=(2,)   hand=['-0.0440' '-0.0517'] NumPy=['-0.0440' '-0.0517']
dL_dW1       shape=(2, 2) hand=[['-0.0440' '0.0000']
 ['-0.0517' '0.0000']] NumPy=[['-0.0440' '0.0000']
 ['-0.0517' '0.0000']]
dL_db1       shape=(2,)   hand=['-0.0440' '-0.0517'] NumPy=['-0.0440' '-0.0517']
dL_db2       shape=()     hand=-0.3527 NumPy=-0.3527
All 13 hand-reference checkpoints agree; maximum absolute error=2.220e-16


## 5. 전체 매개변수의 독립 수치 검증

매개변수를 다음 순서로 `(9,)` 벡터로 펼친다.

`[W1[0,0], W1[0,1], W1[1,0], W1[1,1], b1[0], b1[1], W2[0], W2[1], b2]`

각 좌표 $i$에서 $[L(\theta+he_i)-L(\theta-he_i)]/(2h)$를 계산한다. 이 경로는 위 역전파식을 사용하지 않고 **순전파 손실만** 평가한다.
`h=1e-5`, `atol=1e-6`, `rtol=0`을 적용한다. 하나의 고정 예제에서 모든 9개 성분을 교차 검증하는 것이며 임의의 신경망/입력에 대한 정확도 보장을 뜻하지 않는다.

In [7]:
parameter_order = ["W1[0,0]", "W1[0,1]", "W1[1,0]", "W1[1,1]",
                   "b1[0]", "b1[1]", "W2[0]", "W2[1]", "b2"]
parameters_flat = np.concatenate((W1.ravel(), b1, W2, [b2]))
analytical_gradient = np.concatenate((dW1.ravel(), db1, dW2, [db2]))


def loss_from_flat(parameters):
    """Return this fixed example's scalar BCE from nine finite float64 parameters.

    Input shape is (9,) in parameter_order. Only forward propagation and the
    stable logit BCE are evaluated; no backward formula is used. Bad shape
    or nonfinite parameter input raises ValueError. Input is not modified.
    """
    values = np.asarray(parameters, dtype=np.float64)
    if values.shape != (9,) or not np.all(np.isfinite(values)):
        raise ValueError("parameters must have shape (9,) with finite values")
    weights1 = values[:4].reshape(2, 2)
    bias1 = values[4:6]
    weights2 = values[6:8]
    bias2 = values[8]
    hidden = sigmoid(weights1 @ x + bias1)
    logit = weights2 @ hidden + bias2
    return float(np.logaddexp(0.0, logit) - y_true*logit)


derivative_h = 1e-5
numerical_gradient_values = numerical_gradient(loss_from_flat, parameters_flat, h=derivative_h)
assert analytical_gradient.shape == (9,) and numerical_gradient_values.shape == (9,)
np.testing.assert_allclose(numerical_gradient_values, analytical_gradient, atol=1e-6, rtol=0)
np.testing.assert_array_equal(numerical_gradient_values[[1, 3]], [0.0, 0.0])
max_numerical_gradient_error = float(np.max(np.abs(numerical_gradient_values-analytical_gradient)))
for name, analytic, numeric in zip(parameter_order, analytical_gradient, numerical_gradient_values):
    print(f"{name:9} analytic={analytic: .12f} numerical={numeric: .12f} abs_error={abs(analytic-numeric):.3e}")
print(f"All 9 parameter gradients pass; maximum absolute error={max_numerical_gradient_error:.3e}")

W1[0,0]   analytic=-0.043978515809 numerical=-0.043978515812 abs_error=3.696e-12
W1[0,1]   analytic=-0.000000000000 numerical= 0.000000000000 abs_error=0.000e+00
W1[1,0]   analytic=-0.051733504391 numerical=-0.051733504391 abs_error=5.457e-13
W1[1,1]   analytic=-0.000000000000 numerical= 0.000000000000 abs_error=0.000e+00
b1[0]     analytic=-0.043978515809 numerical=-0.043978515812 abs_error=3.696e-12
b1[1]     analytic=-0.051733504391 numerical=-0.051733504391 abs_error=5.457e-13
W2[0]     analytic=-0.185164585001 numerical=-0.185164584998 abs_error=3.022e-12
W2[1]     analytic=-0.202610718233 numerical=-0.202610718225 abs_error=7.843e-12
b2        analytic=-0.352708430005 numerical=-0.352708430007 abs_error=1.627e-12
All 9 parameter gradients pass; maximum absolute error=7.843e-12


## 6. 확인한 내용

순전파의 곱 방향과 shape, 필수6개 미분과 편향 기울기, $dW_1$의 outer product 구조를 확인했다.
독립 손계산 상수와 NumPy의 원소별 오차·네 자리 표시를 비교했고 모든9개 매개변수를 중심차분으로 다시 검증했다.
검증 결과는 아래 JSON에 원시 정밀도로 저장한다. 사용자 함수 또는 파일 쓰기 오류는 숨기지 않는다.

다음 단계에서는 Gaussian/Bernoulli/categorical likelihood에서 손실함수가 나오는 과정을 별도로 유도한다. 이 노트북은 단일 샘플의 고정 네트워크 검증에 범위를 한정한다.

In [8]:
report = {
    "seed": 42, "dtype": "float64", "numpy_version": np.__version__,
    "parameter_count": 9, "parameter_order": parameter_order,
    "derivative_h": derivative_h,
    "max_hand_absolute_error": max_hand_absolute_error,
    "max_numerical_gradient_error": max_numerical_gradient_error,
    "four_decimal_match": four_decimal_match, "all_assertions_passed": True,
    "parameters": {name: np.asarray(value).tolist() for name, value in
                   {"x": x, "W1": W1, "b1": b1, "W2": W2, "b2": b2, "y_true": y_true}.items()},
    "forward": {name: np.asarray(value).tolist() for name, value in forward.items()},
    "backward": {name: np.asarray(value).tolist() for name, value in backward.items()},
    "hand_reference": {name: np.asarray(value).tolist() for name, value in hand_reference.items()},
    "shapes": {name: list(np.shape(value)) for name, value in computed.items()},
    "hand_absolute_errors": hand_errors,
    "analytical_gradient": analytical_gradient.tolist(),
    "numerical_gradient": numerical_gradient_values.tolist(),
    "notes": [
        "One sample; no batch mean factor; natural logarithms.",
        "Hand-reference literals are independent of the NumPy forward/backward computation.",
        "Only four-decimal display normalizes signed zero; original arrays are preserved.",
        "The probability BCE and stable logit BCE agree for this nonsaturated fixed example.",
    ],
}
(output_dir / "backprop_metrics.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False)+"\n",
    encoding="utf-8", newline="\n",
)
print("PASS: 13 hand-reference checkpoints, 4-decimal agreement, all 9 numerical gradients")
print("Saved backprop_metrics.json")

PASS: 13 hand-reference checkpoints, 4-decimal agreement, all 9 numerical gradients
Saved backprop_metrics.json
